# 02 – Batch processing et optimisation

In [ ]:
import re, time
from operator import add
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, BooleanType)
from shopstream_utils import get_spark, read_table, write_table, JDBC_URL, JDBC_PROPS, DATA_DIR

spark = get_spark("02-batch")

print("Version Spark :", spark.version)
print("Application :", spark.sparkContext.appName)
print("Spark UI :", spark.sparkContext.uiWebUrl)
print("Parallélisme par défaut :", spark.sparkContext.defaultParallelism)
print("Dossier des données :", DATA_DIR)

## 3.1 Charger l'historique dans PostgreSQL


Cette partie compare deux méthodes de lecture d'un fichier CSV :

1. lecture avec inférence automatique du schéma ;
2. lecture avec un schéma explicite.

Nous préparons ensuite les colonnes `hashtags`, `event_time` et `text_length`, avant d'enregistrer les avis et les produits dans PostgreSQL.

La durée de chaque lecture est mesurée après une action `count()`, car Spark utilise l'évaluation paresseuse.

### Lecture avec inférence automatique du schéma

Avec `inferSchema=True`, Spark parcourt les données pour déterminer automatiquement le type de chaque colonne.

L'action `count()` force l'exécution de la lecture et permet de mesurer sa durée réelle.

In [7]:
start = time.time()

hist_inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{DATA_DIR}/reviews_history.csv")
)

nb_reviews_inferred = hist_inferred.count()
duration_infer = time.time() - start

print("Nombre d'avis :", nb_reviews_inferred)
print(f"Durée avec inferSchema : {duration_infer:.3f} secondes")

hist_inferred.printSchema()

Nombre d'avis : 200000
Durée avec inferSchema : 3.557 secondes
root
 |-- review_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- lang: string (nullable = true)
 |-- text: string (nullable = true)
 |-- hashtags: string (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- country: string (nullable = true)



In [8]:
hist_inferred.show(5, truncate=False)

+------------------------------------+-------------------+-------+---------+----------+------------+------+----+----------------------------------------------------------------+-------------------------------+-------------+-----------------+-------+
|review_id                           |event_time         |user_id|username |product_id|category    |rating|lang|text                                                            |hashtags                       |helpful_votes|verified_purchase|country|
+------------------------------------+-------------------+-------+---------+----------+------------+------+----+----------------------------------------------------------------+-------------------------------+-------------+-----------------+-------+
|db8ffac6-9626-c1fb-5d0a-02058015b675|2026-07-14 14:03:17|u00139 |adam_139 |p0037     |electronique|4     |fr  |Tres satisfait de ce chargeur :( #electronique #happy #nouveaute|#electronique|#happy|#nouveaute|1            |true             |FR     |


### Lecture avec un schéma explicite

Avec un schéma explicite, Spark connaît les types avant de lire les données. Il n'a donc pas besoin de parcourir préalablement le fichier pour les deviner.

La colonne `event_time` est d'abord lue comme une chaîne de caractères. Elle sera ensuite convertie en timestamp.

La colonne `hashtags` est également lue comme une chaîne avant d'être transformée en tableau.

In [10]:
history_schema = StructType([
    StructField("review_id", StringType(), False),
    StructField("event_time", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("username", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("rating", IntegerType(), True),
    StructField("lang", StringType(), True),
    StructField("text", StringType(), True),
    StructField("hashtags", StringType(), True),
    StructField("helpful_votes", IntegerType(), True),
    StructField("verified_purchase", BooleanType(), True),
    StructField("country", StringType(), True)
])

In [11]:
start = time.time()

hist_raw = (
    spark.read
    .option("header", True)
    .schema(history_schema)
    .csv(f"{DATA_DIR}/reviews_history.csv")
)

nb_reviews_explicit = hist_raw.count()
duration_explicit = time.time() - start

print("Nombre d'avis :", nb_reviews_explicit)
print(f"Durée avec schéma explicite : {duration_explicit:.3f} secondes")
print(f"Durée avec inferSchema      : {duration_infer:.3f} secondes")

hist_raw.printSchema()

Nombre d'avis : 200000
Durée avec schéma explicite : 0.252 secondes
Durée avec inferSchema      : 3.557 secondes
root
 |-- review_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- lang: string (nullable = true)
 |-- text: string (nullable = true)
 |-- hashtags: string (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- country: string (nullable = true)



In [12]:
comparison_schema = spark.createDataFrame(
    [
        ("inferSchema", duration_infer),
        ("schéma explicite", duration_explicit)
    ],
    ["methode", "duree_secondes"]
)

comparison_schema.show(truncate=False)

+----------------+------------------+
|methode         |duree_secondes    |
+----------------+------------------+
|inferSchema     |3.55719256401062  |
|schéma explicite|0.2519824504852295|
+----------------+------------------+



### Préparation des colonnes

Les transformations suivantes sont appliquées :

- `hashtags` est convertie d'une chaîne séparée par `|` vers un tableau ;
- `event_time` est convertie en timestamp ;
- `text_length` contient le nombre de caractères du texte.

Le caractère `|` possède une signification particulière dans une expression régulière. Il doit donc être échappé avec `\|`.

In [13]:
hist_prepared = (
    hist_raw
    .withColumn(
        "hashtags",
        F.when(
            F.col("hashtags").isNull() | (F.trim(F.col("hashtags")) == ""),
            F.array().cast("array<string>")
        ).otherwise(
            F.split(F.col("hashtags"), r"\|")
        )
    )
    .withColumn(
        "event_time",
        F.to_timestamp("event_time")
    )
    .withColumn(
        "text_length",
        F.length("text")
    )
)

In [14]:
hist_prepared.printSchema()

hist_prepared.select(
    "review_id",
    "event_time",
    "hashtags",
    "text",
    "text_length"
).show(10, truncate=False)

root
 |-- review_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- lang: string (nullable = true)
 |-- text: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- helpful_votes: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- country: string (nullable = true)
 |-- text_length: integer (nullable = true)

+------------------------------------+-------------------+-----------------------------------+------------------------------------------------------------------------+-----------+
|review_id                           |event_time         |hashtags                           |text                                                                    |text_length|
+---------

In [15]:
hist_prepared.select(
    F.count("*").alias("total"),
    F.sum(F.col("review_id").isNull().cast("int")).alias("review_id_null"),
    F.sum(F.col("event_time").isNull().cast("int")).alias("event_time_null"),
    F.sum(F.col("text").isNull().cast("int")).alias("text_null")
).show()

+------+--------------+---------------+---------+
| total|review_id_null|event_time_null|text_null|
+------+--------------+---------------+---------+
|200000|             0|              0|        0|
+------+--------------+---------------+---------+



### Chargement du référentiel des produits

Le fichier `products.csv` contient les informations descriptives utilisées plus tard pour enrichir les avis, notamment lors de l'analyse par marque.

In [16]:
products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{DATA_DIR}/products.csv")
)

print("Nombre de produits :", products.count())
products.printSchema()
products.show(10, truncate=False)

Nombre de produits : 200
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)

+----------+----------------+------------+-----+------+
|product_id|product_name    |category    |brand|price |
+----------+----------------+------------+-----+------+
|p0000     |couette plus    |maison      |Orbis|18.28 |
|p0001     |peluche classic |jouets      |Kairo|24.12 |
|p0002     |raquette pro    |sport       |Nova |80.51 |
|p0003     |shampoing eco   |beaute      |Brume|7.8   |
|p0004     |tapis max       |sport       |Atlas|56.48 |
|p0005     |miel pro        |alimentation|Kairo|6.42  |
|p0006     |peluche mini    |jouets      |Orbis|58.57 |
|p0007     |casque classic  |electronique|Nova |47.43 |
|p0008     |chocolat classic|alimentation|Brume|144.05|
|p0009     |souris mini     |electronique|Zenit|315.35|
+----------+----------------+------------

### Écriture dans PostgreSQL

Les avis préparés sont enregistrés dans la table `reviews_history`.

Le référentiel des produits est enregistré dans la table `products`.

Le mode `overwrite` remplace les tables si elles existent déjà.

In [17]:
write_table(
    hist_prepared,
    "reviews_history",
    mode="overwrite"
)

write_table(
    products,
    "products",
    mode="overwrite"
)

print("OK - tables reviews_history et products écrites dans PostgreSQL")

OK - tables reviews_history et products écrites dans PostgreSQL


### Relecture depuis PostgreSQL

Pour la suite du TP, le DataFrame `hist` provient de PostgreSQL et non directement du fichier CSV.

Cela permet de travailler dans les mêmes conditions qu'une application utilisant une base de données comme source.

In [18]:
hist = read_table(spark, "reviews_history")
products_db = read_table(spark, "products")

print("Nombre d'avis relus :", hist.count())
print("Nombre de produits relus :", products_db.count())

print("Nombre de partitions de hist :", hist.rdd.getNumPartitions())

Nombre d'avis relus : 200000
Nombre de produits relus : 200
Nombre de partitions de hist : 1


In [19]:
hist.printSchema()
hist.show(5, truncate=False)

root
 |-- review_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- lang: string (nullable = true)
 |-- text: string (nullable = true)
 |-- hashtags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- helpful_votes: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- country: string (nullable = true)
 |-- text_length: integer (nullable = true)

+------------------------------------+-------------------+-------+---------+----------+------------+------+----+----------------------------------------------------------------+-----------------------------------+-------------+-----------------+-------+-----------+
|review_id                           |event_time         |user_id|username |product_id|category    |rati

### Q3.1 — Pourquoi `inferSchema` est-il plus lent ?

`inferSchema=True` est plus lent parce que Spark doit parcourir les données afin de déterminer automatiquement le type de chaque colonne avant d'effectuer le traitement demandé.

Avec un schéma explicite, les types sont déjà fournis. Spark peut donc lire directement les données sans effectuer cette phase d'inférence.

D'après la Spark UI :

- la lecture avec `inferSchema=True`, suivie de `count()`, a déclenché **4 jobs** : deux jobs pour l'analyse du CSV et deux jobs pour l'action `count()` ;
- la lecture avec un schéma explicite, suivie de `count()`, a déclenché **2 jobs**, correspondant à l'exécution du `count()`.

Les jobs `showString` ne sont pas inclus, car ils proviennent des appels à `show()` utilisés uniquement pour afficher les résultats.

## 3.2 Retour aux RDD – 15 mots les plus fréquents

## 3.2 Analyse des mots avec les RDD

L'objectif est de calculer les 15 mots les plus fréquents dans les textes des avis.

Les traitements appliqués sont :

1. suppression des hashtags ;
2. conversion du texte en minuscules ;
3. découpage du texte en mots ;
4. conservation des mots comportant au moins quatre caractères ;
5. association de chaque mot à la valeur 1 ;
6. addition des occurrences par mot ;
7. sélection des 15 mots les plus fréquents.

Le calcul est d'abord réalisé avec l'API RDD, puis avec l'API DataFrame.

In [20]:
def extract_words(text):
    if text is None:
        return []

    text_without_hashtags = re.sub(r"#\w+", "", text.lower())

    return re.findall(
        r"\b[a-zA-ZÀ-ÿ]{4,}\b",
        text_without_hashtags
    )

In [21]:
example = "Très bon produit #promotion et livraison rapide !"

extract_words(example)

['très', 'produit', 'livraison', 'rapide']

In [22]:
words_rdd = (
    hist
    .select("text")
    .rdd
    .flatMap(lambda row: extract_words(row["text"]))
    .filter(lambda word: len(word) >= 4)
    .map(lambda word: (word, 1))
)

word_counts_rdd = words_rdd.reduceByKey(add)

In [23]:
start = time.time()

top15_rdd = word_counts_rdd.takeOrdered(
    15,
    key=lambda item: (-item[1], item[0])
)

duration_rdd = time.time() - start

print(f"Durée RDD : {duration_rdd:.3f} secondes")

for word, count in top15_rdd:
    print(f"{word:<20} {count}")

Durée RDD : 1.466 secondes
pour                 40420
produit              29651
prix                 28918
tres                 26091
qualite              25594
conforme             24648
apres                22185
service              21361
livraison            18521
honnetement          18014
semaine              17925
utilisation          17925
cadeau               17819
vraiment             17787
excellent            15311


### Calcul avec l'API DataFrame

Le même traitement est maintenant réalisé avec les fonctions natives de Spark SQL.

Les hashtags sont supprimés avec `regexp_replace`. Le texte est converti en minuscules, découpé en mots, puis les mots sont éclatés avec `explode`.

Les fonctions natives sont préférables aux UDF Python, car Spark peut analyser et optimiser leur plan d'exécution avec Catalyst.

In [24]:
words_df = (
    hist
    .select(
        F.explode(
            F.split(
                F.regexp_replace(
                    F.lower(F.col("text")),
                    r"#\w+",
                    ""
                ),
                r"[^a-zA-ZÀ-ÿ]+"
            )
        ).alias("word")
    )
    .filter(F.length("word") >= 4)
)

In [25]:
words_df.show(20, truncate=False)

+-----------+
|word       |
+-----------+
|tres       |
|satisfait  |
|chargeur   |
|asked      |
|refund     |
|honestly   |
|magnifique |
|conforme   |
|photos     |
|tres       |
|produit    |
|scam       |
|nothing    |
|like       |
|pictures   |
|gift       |
|produit    |
|arrive     |
|casse      |
|honnetement|
+-----------+
only showing top 20 rows



In [26]:
word_counts_df = (
    words_df
    .groupBy("word")
    .agg(F.count("*").alias("count"))
)

In [27]:
top15_df = (
    word_counts_df
    .orderBy(
        F.desc("count"),
        F.asc("word")
    )
    .limit(15)
)

start = time.time()

top15_df_result = top15_df.collect()

duration_df = time.time() - start

print(f"Durée DataFrame : {duration_df:.3f} secondes")

for row in top15_df_result:
    print(f"{row['word']:<20} {row['count']}")

Durée DataFrame : 1.170 secondes
pour                 40420
produit              29651
prix                 28918
tres                 26091
qualite              25594
conforme             24648
apres                22185
service              21361
livraison            18521
honnetement          18014
semaine              17925
utilisation          17925
cadeau               17819
vraiment             17787
excellent            15311


In [28]:
top15_rdd_normalized = [
    (word, count)
    for word, count in top15_rdd
]

top15_df_normalized = [
    (row["word"], row["count"])
    for row in top15_df_result
]

print("Top 15 RDD :")
print(top15_rdd_normalized)

print("\nTop 15 DataFrame :")
print(top15_df_normalized)

print("\nRésultats identiques :", top15_rdd_normalized == top15_df_normalized)

Top 15 RDD :
[('pour', 40420), ('produit', 29651), ('prix', 28918), ('tres', 26091), ('qualite', 25594), ('conforme', 24648), ('apres', 22185), ('service', 21361), ('livraison', 18521), ('honnetement', 18014), ('semaine', 17925), ('utilisation', 17925), ('cadeau', 17819), ('vraiment', 17787), ('excellent', 15311)]

Top 15 DataFrame :
[('pour', 40420), ('produit', 29651), ('prix', 28918), ('tres', 26091), ('qualite', 25594), ('conforme', 24648), ('apres', 22185), ('service', 21361), ('livraison', 18521), ('honnetement', 18014), ('semaine', 17925), ('utilisation', 17925), ('cadeau', 17819), ('vraiment', 17787), ('excellent', 15311)]

Résultats identiques : True


In [29]:
top15_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (13)
+- == Final Plan ==
   TakeOrderedAndProject (9)
   +- * HashAggregate (8)
      +- AQEShuffleRead (7)
         +- ShuffleQueryStage (6), Statistics(sizeInBytes=7.3 KiB, rowCount=227)
            +- Exchange (5)
               +- * HashAggregate (4)
                  +- * Filter (3)
                     +- * Generate (2)
                        +- * Scan JDBCRelation(reviews_history) [numPartitions=1]  (1)
+- == Initial Plan ==
   TakeOrderedAndProject (12)
   +- HashAggregate (11)
      +- Exchange (10)
         +- HashAggregate (4)
            +- Filter (3)
               +- Generate (2)
                  +- Scan JDBCRelation(reviews_history) [numPartitions=1]  (1)


(1) Scan JDBCRelation(reviews_history) [numPartitions=1]  [codegen id : 1]
Output [1]: [text#456]
ReadSchema: struct<text:string>

(2) Generate [codegen id : 1]
Input [1]: [text#456]
Arguments: explode(split(regexp_replace(lower(text#456), #\w+, , 1), [^a-zA-ZÀ-ÿ]+, -1)), false,

### Q3.2 — Transformations narrow et wide

Dans la chaîne RDD :

- `flatMap` est une transformation narrow ;
- `filter` est une transformation narrow ;
- `map` est une transformation narrow ;
- `reduceByKey` est une transformation wide ;
- `takeOrdered` est une action.

Les transformations narrow peuvent traiter chaque partition indépendamment sans redistribuer les données.

`reduceByKey` est wide, car toutes les occurrences d'un même mot doivent être regroupées. Cette opération provoque donc un shuffle.

La Spark UI montre deux stages :

1. le premier stage exécute notamment `flatMap`, `filter`, `map` et la première partie de `reduceByKey`. Il écrit 2,7 KiB de données de shuffle ;
2. le deuxième stage lit les 2,7 KiB du shuffle et exécute `takeOrdered`.

Le shuffle provoqué par `reduceByKey` constitue la frontière entre les deux stages.

### Q3.3 — Pourquoi rien n'est calculé avant `takeOrdered` ?

Ce principe s'appelle la lazy evaluation, ou évaluation paresseuse.

Spark enregistre les transformations `flatMap`, `filter`, `map` et `reduceByKey`, mais ne les exécute pas immédiatement.

L'appel à l'action `takeOrdered` déclenche la construction, l'optimisation et l'exécution du plan complet.

Ce fonctionnement permet à Spark d'optimiser les opérations, d'éviter certains calculs inutiles et d'organiser efficacement leur exécution sur les partitions.

## 3.3 Transformations et agrégations

### A. Utilisateurs actifs

Un utilisateur actif est défini comme un utilisateur ayant publié au moins 60 avis.

Le traitement suit les étapes suivantes :

1. compter les avis de chaque utilisateur ;
2. filtrer les utilisateurs ayant au moins 60 avis ;
3. joindre cette liste avec l'historique ;
4. calculer le nombre d'utilisateurs actifs ;
5. calculer la part des avis publiée par ces utilisateurs.

In [31]:
user_counts = (
    hist
    .groupBy("user_id")
    .agg(
        F.count("*").alias("nb_reviews")
    )
)

user_counts.orderBy(
    F.desc("nb_reviews")
).show(20, truncate=False)

+-------+----------+
|user_id|nb_reviews|
+-------+----------+
|u01967 |16637     |
|u01239 |8855      |
|u00184 |6181      |
|u01580 |4898      |
|u00198 |3930      |
|u00410 |3206      |
|u00259 |2844      |
|u01893 |2573      |
|u01431 |2417      |
|u00219 |2158      |
|u01906 |1910      |
|u01896 |1734      |
|u00316 |1642      |
|u01394 |1582      |
|u01892 |1431      |
|u01970 |1401      |
|u01981 |1255      |
|u00212 |1245      |
|u00274 |1164      |
|u00794 |1159      |
+-------+----------+
only showing top 20 rows



In [32]:
active_users = (
    user_counts
    .filter(F.col("nb_reviews") >= 60)
)

active_users.orderBy(
    F.desc("nb_reviews")
).show(20, truncate=False)

+-------+----------+
|user_id|nb_reviews|
+-------+----------+
|u01967 |16637     |
|u01239 |8855      |
|u00184 |6181      |
|u01580 |4898      |
|u00198 |3930      |
|u00410 |3206      |
|u00259 |2844      |
|u01893 |2573      |
|u01431 |2417      |
|u00219 |2158      |
|u01906 |1910      |
|u01896 |1734      |
|u00316 |1642      |
|u01394 |1582      |
|u01892 |1431      |
|u01970 |1401      |
|u01981 |1255      |
|u00212 |1245      |
|u00274 |1164      |
|u00794 |1159      |
+-------+----------+
only showing top 20 rows



In [33]:
nb_active_users = active_users.count()

active_reviews = (
    hist
    .join(
        active_users.select("user_id"),
        on="user_id",
        how="inner"
    )
)

nb_active_reviews = active_reviews.count()
nb_total_reviews = hist.count()

active_reviews_share = (
    nb_active_reviews / nb_total_reviews * 100
    if nb_total_reviews > 0
    else 0
)

print("Nombre d'utilisateurs actifs :", nb_active_users)
print("Nombre total d'avis :", nb_total_reviews)
print("Avis publiés par les utilisateurs actifs :", nb_active_reviews)
print(f"Part des avis publiée par les utilisateurs actifs : {active_reviews_share:.2f} %")

Nombre d'utilisateurs actifs : 529
Nombre total d'avis : 200000
Avis publiés par les utilisateurs actifs : 155770
Part des avis publiée par les utilisateurs actifs : 77.89 %


#### Résultat

L'historique contient **529 utilisateurs actifs**, c'est-à-dire des utilisateurs ayant publié au moins 60 avis.

Ces utilisateurs ont publié **155770 avis** sur un total de **200 000 avis**, soit **77.8%** de l'historique.

### Version Spark SQL

La table temporaire `reviews_history` permet d'exécuter une requête SQL directement sur le DataFrame `hist`.

La requête regroupe les avis par utilisateur et conserve ceux ayant publié au moins 60 avis avec la clause `HAVING`.

In [35]:
hist.createOrReplaceTempView("reviews_history")

In [36]:
active_users_sql = spark.sql("""
    SELECT
        user_id,
        COUNT(*) AS nb_reviews
    FROM reviews_history
    GROUP BY user_id
    HAVING COUNT(*) >= 60
""")

active_users_sql.orderBy(
    F.desc("nb_reviews")
).show(20, truncate=False)

+-------+----------+
|user_id|nb_reviews|
+-------+----------+
|u01967 |16637     |
|u01239 |8855      |
|u00184 |6181      |
|u01580 |4898      |
|u00198 |3930      |
|u00410 |3206      |
|u00259 |2844      |
|u01893 |2573      |
|u01431 |2417      |
|u00219 |2158      |
|u01906 |1910      |
|u01896 |1734      |
|u00316 |1642      |
|u01394 |1582      |
|u01892 |1431      |
|u01970 |1401      |
|u01981 |1255      |
|u00212 |1245      |
|u00274 |1164      |
|u00794 |1159      |
+-------+----------+
only showing top 20 rows



In [38]:
print("DataFrame :", active_users.count())
print("Spark SQL :", active_users_sql.count())

DataFrame : 529
Spark SQL : 529


In [39]:
print("PLAN DATAFRAME")
active_users.explain("formatted")

print("\nPLAN SPARK SQL")
active_users_sql.explain("formatted")

PLAN DATAFRAME
== Physical Plan ==
AdaptiveSparkPlan (6)
+- Filter (5)
   +- HashAggregate (4)
      +- Exchange (3)
         +- HashAggregate (2)
            +- Scan JDBCRelation(reviews_history) [numPartitions=1]  (1)


(1) Scan JDBCRelation(reviews_history) [numPartitions=1] 
Output [1]: [user_id#450]
ReadSchema: struct<user_id:string>

(2) HashAggregate
Input [1]: [user_id#450]
Keys [1]: [user_id#450]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#612L]
Results [2]: [user_id#450, count#613L]

(3) Exchange
Input [2]: [user_id#450, count#613L]
Arguments: hashpartitioning(user_id#450, 8), ENSURE_REQUIREMENTS, [plan_id=1145]

(4) HashAggregate
Input [2]: [user_id#450, count#613L]
Keys [1]: [user_id#450]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#602L]
Results [2]: [user_id#450, count(1)#602L AS nb_reviews#603L]

(5) Filter
Input [2]: [user_id#450, nb_reviews#603L]
Condition : (nb_reviews#603L >= 60)

(6) AdaptiveSparkPlan
Output [2]: [user_id#450,

#### Comparaison DataFrame et SQL

Les versions DataFrame et SQL produisent le même résultat.

Leurs plans physiques sont très proches, car les deux API utilisent le même moteur d'optimisation Catalyst. Le choix entre DataFrame et SQL concerne donc principalement la lisibilité et les préférences de développement, plutôt qu'une différence automatique de performances.

#### Résultat — Utilisateurs actifs

L'historique contient **529 utilisateurs actifs**, c'est-à-dire des utilisateurs ayant publié au moins 60 avis.

Ces utilisateurs ont publié **155 770 avis** sur un total de **200 000 avis**, soit **77,89 %** de l'historique.

L'activité est donc fortement concentrée : les utilisateurs actifs produisent plus des trois quarts des avis.

### B. Nombre d'avis par jour

Cette analyse calcule le nombre total d'avis publiés chaque jour afin d'identifier une éventuelle journée anormale.

In [40]:
daily_counts = (
    hist
    .withColumn("day", F.to_date("event_time"))
    .groupBy("day")
    .agg(F.count("*").alias("nb_reviews"))
)

In [41]:
daily_counts.orderBy("day").show(100, truncate=False)

+----------+----------+
|day       |nb_reviews|
+----------+----------+
|2026-06-25|2141      |
|2026-06-26|2079      |
|2026-06-27|2104      |
|2026-06-28|2045      |
|2026-06-29|2083      |
|2026-06-30|2087      |
|2026-07-01|2095      |
|2026-07-02|2097      |
|2026-07-03|2140      |
|2026-07-04|2149      |
|2026-07-05|2115      |
|2026-07-06|2140      |
|2026-07-07|2084      |
|2026-07-08|2130      |
|2026-07-09|2133      |
|2026-07-10|2097      |
|2026-07-11|2083      |
|2026-07-12|2097      |
|2026-07-13|2092      |
|2026-07-14|2009      |
|2026-07-15|2050      |
|2026-07-16|2048      |
|2026-07-17|2031      |
|2026-07-18|2141      |
|2026-07-19|2113      |
|2026-07-20|2072      |
|2026-07-21|2013      |
|2026-07-22|2077      |
|2026-07-23|2103      |
|2026-07-24|2080      |
|2026-07-25|14123     |
|2026-07-26|2073      |
|2026-07-27|2074      |
|2026-07-28|2068      |
|2026-07-29|2171      |
|2026-07-30|2099      |
|2026-07-31|2099      |
|2026-08-01|2137      |
|2026-08-02|2048

In [42]:
daily_counts.orderBy(
    F.desc("nb_reviews")
).show(10, truncate=False)

+----------+----------+
|day       |nb_reviews|
+----------+----------+
|2026-07-25|14123     |
|2026-09-03|2196      |
|2026-09-16|2191      |
|2026-07-29|2171      |
|2026-09-17|2166      |
|2026-09-14|2164      |
|2026-09-19|2158      |
|2026-09-09|2153      |
|2026-08-25|2151      |
|2026-07-04|2149      |
+----------+----------+
only showing top 10 rows



In [43]:
daily_stats = daily_counts.agg(
    F.avg("nb_reviews").alias("mean_reviews"),
    F.stddev("nb_reviews").alias("std_reviews")
).first()

mean_reviews = daily_stats["mean_reviews"]
std_reviews = daily_stats["std_reviews"]

daily_with_score = (
    daily_counts
    .withColumn(
        "z_score",
        (F.col("nb_reviews") - mean_reviews) / std_reviews
    )
)

print(f"Moyenne quotidienne : {mean_reviews:.2f}")
print(f"Écart-type : {std_reviews:.2f}")

daily_with_score.orderBy(
    F.desc("z_score")
).show(10, truncate=False)

Moyenne quotidienne : 2222.22
Écart-type : 1269.28
+----------+----------+---------------------+
|day       |nb_reviews|z_score              |
+----------+----------+---------------------+
|2026-07-25|14123     |9.376024853298562    |
|2026-09-03|2196      |-0.020659171346209457|
|2026-09-16|2191      |-0.0245984201198511  |
|2026-07-29|2171      |-0.04035541521441766 |
|2026-09-17|2166      |-0.044294663988059306|
|2026-09-14|2164      |-0.04587036349751596 |
|2026-09-19|2158      |-0.050597462025885935|
|2026-09-09|2153      |-0.05453671079952758 |
|2026-08-25|2151      |-0.05611241030898423 |
|2026-07-04|2149      |-0.057688109818440886|
+----------+----------+---------------------+
only showing top 10 rows



In [44]:
anomalous_days = (
    daily_with_score
    .filter(F.abs("z_score") > 3)
    .orderBy(F.desc("z_score"))
)

anomalous_days.show(truncate=False)

+----------+----------+-----------------+
|day       |nb_reviews|z_score          |
+----------+----------+-----------------+
|2026-07-25|14123     |9.376024853298562|
+----------+----------+-----------------+



In [45]:
daily_counts_sql = spark.sql("""
    SELECT
        TO_DATE(event_time) AS day,
        COUNT(*) AS nb_reviews
    FROM reviews_history
    GROUP BY TO_DATE(event_time)
    ORDER BY day
""")

daily_counts_sql.show(100, truncate=False)

+----------+----------+
|day       |nb_reviews|
+----------+----------+
|2026-06-25|2141      |
|2026-06-26|2079      |
|2026-06-27|2104      |
|2026-06-28|2045      |
|2026-06-29|2083      |
|2026-06-30|2087      |
|2026-07-01|2095      |
|2026-07-02|2097      |
|2026-07-03|2140      |
|2026-07-04|2149      |
|2026-07-05|2115      |
|2026-07-06|2140      |
|2026-07-07|2084      |
|2026-07-08|2130      |
|2026-07-09|2133      |
|2026-07-10|2097      |
|2026-07-11|2083      |
|2026-07-12|2097      |
|2026-07-13|2092      |
|2026-07-14|2009      |
|2026-07-15|2050      |
|2026-07-16|2048      |
|2026-07-17|2031      |
|2026-07-18|2141      |
|2026-07-19|2113      |
|2026-07-20|2072      |
|2026-07-21|2013      |
|2026-07-22|2077      |
|2026-07-23|2103      |
|2026-07-24|2080      |
|2026-07-25|14123     |
|2026-07-26|2073      |
|2026-07-27|2074      |
|2026-07-28|2068      |
|2026-07-29|2171      |
|2026-07-30|2099      |
|2026-07-31|2099      |
|2026-08-01|2137      |
|2026-08-02|2048

In [46]:
df_daily_result = {
    (row["day"], row["nb_reviews"])
    for row in daily_counts.collect()
}

sql_daily_result = {
    (row["day"], row["nb_reviews"])
    for row in daily_counts_sql.collect()
}

print("Résultats identiques :", df_daily_result == sql_daily_result)

Résultats identiques : True


In [47]:
print("PLAN DATAFRAME")
daily_counts.explain("formatted")

print("\nPLAN SQL")
daily_counts_sql.explain("formatted")

PLAN DATAFRAME
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Final Plan ==
   * HashAggregate (7)
   +- AQEShuffleRead (6)
      +- ShuffleQueryStage (5), Statistics(sizeInBytes=2.1 KiB, rowCount=90)
         +- Exchange (4)
            +- * HashAggregate (3)
               +- * Project (2)
                  +- * Scan JDBCRelation(reviews_history) [numPartitions=1]  (1)
+- == Initial Plan ==
   HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan JDBCRelation(reviews_history) [numPartitions=1]  (1)


(1) Scan JDBCRelation(reviews_history) [numPartitions=1]  [codegen id : 1]
Output [1]: [event_time#449]
ReadSchema: struct<event_time:timestamp>

(2) Project [codegen id : 1]
Output [1]: [cast(event_time#449 as date) AS day#730]
Input [1]: [event_time#449]

(3) HashAggregate [codegen id : 1]
Input [1]: [day#730]
Keys [1]: [day#730]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#771L]
Results [2]: [day#730, count#

#### Résultat — Activité quotidienne

Le **25 juillet 2026** est une journée anormale avec **14 123 avis**.

Les autres journées comptent généralement entre 2 000 et 2 200 avis. La moyenne sur l'ensemble de la période est de **2 222,22 avis par jour**, mais elle est tirée vers le haut par cette valeur exceptionnelle.

Le volume du 25 juillet est donc environ six fois supérieur au volume quotidien habituel.

#### Comparaison DataFrame et Spark SQL

Les deux versions produisent exactement les mêmes nombres d'avis par jour.

Leurs plans utilisent une lecture JDBC, une conversion du timestamp en date, une agrégation partielle, un shuffle sur la date, puis une agrégation finale.

Le plan SQL contient une étape supplémentaire de redistribution par plage (`rangepartitioning`) suivie d'un tri global. Cette différence vient de la clause `ORDER BY day` présente dans la requête SQL.

Le DataFrame `daily_counts` ne contient pas directement de tri : `orderBy("day")` a seulement été appliqué au moment de l'affichage.

### C. Top 10 des hashtags

La colonne `hashtags` contient un tableau de hashtags. La fonction `explode` transforme chaque élément du tableau en une ligne distincte.

Les hashtags sont ensuite regroupés et comptés afin d'identifier les dix plus fréquents.

In [48]:
hashtag_counts = (
    hist
    .select(
        F.explode("hashtags").alias("hashtag")
    )
    .filter(
        F.col("hashtag").isNotNull()
        & (F.trim(F.col("hashtag")) != "")
    )
    .groupBy("hashtag")
    .agg(
        F.count("*").alias("nb_occurrences")
    )
)

top10_hashtags = (
    hashtag_counts
    .orderBy(
        F.desc("nb_occurrences"),
        F.asc("hashtag")
    )
    .limit(10)
)

top10_hashtags.show(truncate=False)

+-----------+--------------+
|hashtag    |nb_occurrences|
+-----------+--------------+
|#top       |15395         |
|#qualite   |15370         |
|#bonplan   |15345         |
|#happy     |15344         |
|#recommande|15267         |
|#beaute    |15150         |
|#livraison |15010         |
|#cadeau    |14987         |
|#nouveaute |14961         |
|#prix      |14929         |
+-----------+--------------+



#### Résultat — Hashtags dominants

Le hashtag le plus fréquent est **#top**, avec **15 395 occurrences**.

Il est suivi par **#qualite** avec 15 370 occurrences, **#bonplan** avec 15 345 occurrences et **#happy** avec 15 344 occurrences.

Les volumes des principaux hashtags sont très proches. L'utilisation des hashtags est donc relativement équilibrée et aucun hashtag ne domine fortement l'ensemble de l'historique.

## 3.4 Jointure et broadcast


Les avis sont joints avec le référentiel des produits à partir de `product_id`.

L'objectif est de calculer le nombre d'avis et la note moyenne par marque, puis de comparer trois stratégies :

1. stratégie choisie automatiquement par Spark ;
2. broadcast automatique désactivé ;
3. broadcast forcé de la petite table `products`.

In [50]:
products_db.printSchema()
products_db.show(5, truncate=False)

print("Nombre de produits :", products_db.count())

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)

+----------+---------------+--------+-----+-----+
|product_id|product_name   |category|brand|price|
+----------+---------------+--------+-----+-----+
|p0000     |couette plus   |maison  |Orbis|18.28|
|p0001     |peluche classic|jouets  |Kairo|24.12|
|p0002     |raquette pro   |sport   |Nova |80.51|
|p0003     |shampoing eco  |beaute  |Brume|7.8  |
|p0004     |tapis max      |sport   |Atlas|56.48|
+----------+---------------+--------+-----+-----+
only showing top 5 rows

Nombre de produits : 200


In [51]:
def measure_dataframe(name, dataframe):
    start = time.time()
    rows = dataframe.collect()
    duration = time.time() - start

    print(f"{name} : {duration:.3f} secondes")
    print(f"Nombre de lignes retournées : {len(rows)}")

    return rows, duration

In [52]:
brand_stats_default = (
    hist
    .join(
        products_db,
        on="product_id",
        how="inner"
    )
    .groupBy("brand")
    .agg(
        F.count("*").alias("nb_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
    .orderBy(F.desc("nb_reviews"))
)

In [53]:
default_rows, duration_default = measure_dataframe(
    "Jointure automatique",
    brand_stats_default
)

for row in default_rows:
    print(row)

Jointure automatique : 0.416 secondes
Nombre de lignes retournées : 7
Row(brand='Leto', nb_reviews=72036, avg_rating=3.68)
Row(brand='Atlas', nb_reviews=28318, avg_rating=3.69)
Row(brand='Nova', nb_reviews=27565, avg_rating=3.68)
Row(brand='Zenit', nb_reviews=25110, avg_rating=3.67)
Row(brand='Orbis', nb_reviews=16263, avg_rating=3.68)
Row(brand='Kairo', nb_reviews=15542, avg_rating=3.69)
Row(brand='Brume', nb_reviews=15166, avg_rating=3.69)


In [54]:
brand_stats_default.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (33)
+- == Final Plan ==
   * Sort (21)
   +- AQEShuffleRead (20)
      +- ShuffleQueryStage (19), Statistics(sizeInBytes=280.0 B, rowCount=7)
         +- Exchange (18)
            +- * HashAggregate (17)
               +- AQEShuffleRead (16)
                  +- ShuffleQueryStage (15), Statistics(sizeInBytes=336.0 B, rowCount=7)
                     +- Exchange (14)
                        +- * HashAggregate (13)
                           +- * Project (12)
                              +- * BroadcastHashJoin Inner BuildRight (11)
                                 :- AQEShuffleRead (4)
                                 :  +- ShuffleQueryStage (3), Statistics(sizeInBytes=6.1 MiB, rowCount=2.00E+5)
                                 :     +- Exchange (2)
                                 :        +- * Scan JDBCRelation(reviews_history) [numPartitions=1]  (1)
                                 +- BroadcastQueryStage (10), Statistics(sizeInBytes=2.0 MiB, row

In [55]:
original_broadcast_threshold = spark.conf.get(
    "spark.sql.autoBroadcastJoinThreshold"
)

print(
    "Seuil initial du broadcast :",
    original_broadcast_threshold
)

Seuil initial du broadcast : 10485760b


In [56]:
spark.conf.set(
    "spark.sql.autoBroadcastJoinThreshold",
    -1
)

In [58]:
brand_stats_no_broadcast = (
    hist
    .join(
        products_db,
        on="product_id",
        how="inner"
    )
    .groupBy("brand")
    .agg(
        F.count("*").alias("nb_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
    .orderBy(F.desc("nb_reviews"))
)

In [59]:
no_broadcast_rows, duration_no_broadcast = measure_dataframe(
    "Jointure sans broadcast",
    brand_stats_no_broadcast
)

Jointure sans broadcast : 0.494 secondes
Nombre de lignes retournées : 7


In [60]:
brand_stats_no_broadcast.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (33)
+- == Final Plan ==
   * Sort (21)
   +- AQEShuffleRead (20)
      +- ShuffleQueryStage (19), Statistics(sizeInBytes=280.0 B, rowCount=7)
         +- Exchange (18)
            +- * HashAggregate (17)
               +- AQEShuffleRead (16)
                  +- ShuffleQueryStage (15), Statistics(sizeInBytes=336.0 B, rowCount=7)
                     +- Exchange (14)
                        +- * HashAggregate (13)
                           +- * Project (12)
                              +- * SortMergeJoin Inner (11)
                                 :- * Sort (5)
                                 :  +- AQEShuffleRead (4)
                                 :     +- ShuffleQueryStage (3), Statistics(sizeInBytes=6.1 MiB, rowCount=2.00E+5)
                                 :        +- Exchange (2)
                                 :           +- * Scan JDBCRelation(reviews_history) [numPartitions=1]  (1)
                                 +- * Sort (10)
     

In [62]:
brand_stats_forced = (
    hist
    .join(
        F.broadcast(products_db),
        on="product_id",
        how="inner"
    )
    .groupBy("brand")
    .agg(
        F.count("*").alias("nb_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
    .orderBy(F.desc("nb_reviews"))
)

In [63]:
forced_rows, duration_forced = measure_dataframe(
    "Broadcast forcé",
    brand_stats_forced
)

Broadcast forcé : 0.290 secondes
Nombre de lignes retournées : 7


In [64]:
brand_stats_forced.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (25)
+- == Final Plan ==
   * Sort (15)
   +- AQEShuffleRead (14)
      +- ShuffleQueryStage (13), Statistics(sizeInBytes=280.0 B, rowCount=7)
         +- Exchange (12)
            +- * HashAggregate (11)
               +- AQEShuffleRead (10)
                  +- ShuffleQueryStage (9), Statistics(sizeInBytes=336.0 B, rowCount=7)
                     +- Exchange (8)
                        +- * HashAggregate (7)
                           +- * Project (6)
                              +- * BroadcastHashJoin Inner BuildRight (5)
                                 :- * Scan JDBCRelation(reviews_history) [numPartitions=1]  (1)
                                 +- BroadcastQueryStage (4), Statistics(sizeInBytes=2.0 MiB, rowCount=200)
                                    +- BroadcastExchange (3)
                                       +- * Scan JDBCRelation(products) [numPartitions=1]  (2)
+- == Initial Plan ==
   Sort (24)
   +- Exchange (23)
      +- HashAg

In [65]:
spark.conf.set(
    "spark.sql.autoBroadcastJoinThreshold",
    original_broadcast_threshold
)

In [66]:
join_comparison = spark.createDataFrame(
    [
        ("Automatique", duration_default),
        ("Sans broadcast", duration_no_broadcast),
        ("Broadcast forcé", duration_forced)
    ],
    ["strategie", "duree_secondes"]
)

join_comparison.orderBy("duree_secondes").show(
    truncate=False
)

+---------------+-------------------+
|strategie      |duree_secondes     |
+---------------+-------------------+
|Broadcast forcé|0.2900700569152832 |
|Automatique    |0.41638875007629395|
|Sans broadcast |0.49425363540649414|
+---------------+-------------------+



In [67]:
def identify_join_strategy(dataframe):
    plan = (
        dataframe
        ._jdf
        .queryExecution()
        .executedPlan()
        .toString()
    )

    if "BroadcastHashJoin" in plan:
        return "BroadcastHashJoin"
    elif "SortMergeJoin" in plan:
        return "SortMergeJoin"
    elif "ShuffledHashJoin" in plan:
        return "ShuffledHashJoin"
    else:
        return "Autre stratégie"

print(
    "Automatique :",
    identify_join_strategy(brand_stats_default)
)

print(
    "Sans broadcast :",
    identify_join_strategy(brand_stats_no_broadcast)
)

print(
    "Broadcast forcé :",
    identify_join_strategy(brand_stats_forced)
)

Automatique : BroadcastHashJoin
Sans broadcast : SortMergeJoin
Broadcast forcé : BroadcastHashJoin


### Résultats des jointures

| Stratégie | Durée | Opérateur principal |
|---|---:|---|
| Automatique | 0,416 s | BroadcastHashJoin |
| Broadcast désactivé | 0,494 s | SortMergeJoin |
| Broadcast forcé | 0,290 s | BroadcastHashJoin |

La table `products` est petite par rapport à la table `reviews_history`. Spark choisit donc automatiquement une jointure broadcast.

Lorsque le broadcast est désactivé, Spark utilise une `SortMergeJoin`. Cette stratégie nécessite de redistribuer et de trier les données, ce qui explique ici sa durée plus élevée.

Le broadcast forcé est le plus rapide sur cette exécution, avec 0,290 seconde. Les écarts restent cependant faibles et peuvent varier selon les exécutions, le cache et les ressources disponibles.

## 3.5 Optimisation


Le même ensemble de traitements est exécuté dans plusieurs configurations afin de mesurer l'effet du cache et du partitionnement.

Le workload contient :

- le nombre d'avis par jour, demande B ;
- le top des hashtags, demande C ;
- les moyennes par catégorie et langue, demande E.

Chaque résultat est matérialisé avec `collect()` afin de déclencher réellement les calculs Spark.

In [69]:
def run_workload(df):
    # B : nombre d'avis par jour
    result_b = (
        df
        .withColumn("day", F.to_date("event_time"))
        .groupBy("day")
        .agg(F.count("*").alias("nb_reviews"))
        .orderBy("day")
    )

    # C : top 10 des hashtags
    result_c = (
        df
        .select(
            F.explode("hashtags").alias("hashtag")
        )
        .filter(
            F.col("hashtag").isNotNull()
            & (F.trim(F.col("hashtag")) != "")
        )
        .groupBy("hashtag")
        .agg(F.count("*").alias("nb_occurrences"))
        .orderBy(F.desc("nb_occurrences"))
        .limit(10)
    )

    # E : moyennes par catégorie et langue
    result_e = (
        df
        .groupBy("category", "lang")
        .agg(
            F.avg("text_length").alias("avg_text_length"),
            F.avg("rating").alias("avg_rating")
        )
    )

    result_b.collect()
    result_c.collect()
    result_e.collect()

In [70]:
def measure_workload(name, df):
    start = time.time()

    run_workload(df)

    duration = time.time() - start
    partitions = df.rdd.getNumPartitions()

    print(name)
    print(f"Durée : {duration:.3f} secondes")
    print(f"Nombre de partitions : {partitions}")
    print()

    return duration, partitions

In [71]:
hist.unpersist(blocking=True)
spark.catalog.clearCache()

In [72]:
duration_no_cache, partitions_no_cache = measure_workload(
    "PostgreSQL sans cache",
    hist
)

PostgreSQL sans cache
Durée : 0.663 secondes
Nombre de partitions : 1



In [73]:
hist_cached = hist.cache()

start = time.time()
hist_cached.count()
duration_materialization = time.time() - start

print(
    f"Durée de matérialisation du cache : "
    f"{duration_materialization:.3f} secondes"
)

Durée de matérialisation du cache : 1.645 secondes


In [74]:
duration_cache, partitions_cache = measure_workload(
    "Cache matérialisé",
    hist_cached
)

Cache matérialisé
Durée : 0.448 secondes
Nombre de partitions : 1



In [75]:
hist_cached.unpersist(blocking=True)

DataFrame[review_id: string, event_time: timestamp, user_id: string, username: string, product_id: string, category: string, rating: int, lang: string, text: string, hashtags: array<string>, helpful_votes: int, verified_purchase: boolean, country: string, text_length: int]

In [76]:
hist_repartitioned = (
    hist
    .repartition(8, "category")
    .cache()
)

hist_repartitioned.count()

print(
    "Partitions après repartition :",
    hist_repartitioned.rdd.getNumPartitions()
)

Partitions après repartition : 8


In [77]:
duration_repartition, partitions_repartition = measure_workload(
    "Repartition(8, category) avec cache",
    hist_repartitioned
)

Repartition(8, category) avec cache
Durée : 0.378 secondes
Nombre de partitions : 8



In [78]:
hist_repartitioned.unpersist(blocking=True)

DataFrame[review_id: string, event_time: timestamp, user_id: string, username: string, product_id: string, category: string, rating: int, lang: string, text: string, hashtags: array<string>, helpful_votes: int, verified_purchase: boolean, country: string, text_length: int]

### Comparaison des écritures avec une partition

`coalesce(1)` réduit le nombre de partitions en évitant généralement un shuffle complet.

`repartition(1)` redistribue toutes les données vers une nouvelle partition et provoque donc un shuffle.

In [80]:
daily_result_for_write = (
    hist
    .withColumn("day", F.to_date("event_time"))
    .groupBy("day")
    .agg(F.count("*").alias("nb_reviews"))
)

In [81]:
start = time.time()

(
    daily_result_for_write
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet("/tmp/shopstream_daily_coalesce")
)

duration_coalesce = time.time() - start

print(f"coalesce(1) : {duration_coalesce:.3f} secondes")

coalesce(1) : 0.534 secondes


In [82]:
start = time.time()

(
    daily_result_for_write
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet("/tmp/shopstream_daily_repartition")
)

duration_repartition_one = time.time() - start

print(
    f"repartition(1) : "
    f"{duration_repartition_one:.3f} secondes"
)

repartition(1) : 0.208 secondes


In [83]:
optimization_results = spark.createDataFrame(
    [
        (
            "PostgreSQL sans cache",
            duration_no_cache,
            partitions_no_cache
        ),
        (
            "Cache matérialisé",
            duration_cache,
            partitions_cache
        ),
        (
            "Repartition(8, category) + cache",
            duration_repartition,
            partitions_repartition
        ),
        (
            "Écriture coalesce(1)",
            duration_coalesce,
            1
        ),
        (
            "Écriture repartition(1)",
            duration_repartition_one,
            1
        )
    ],
    [
        "configuration",
        "duree_secondes",
        "nb_partitions"
    ]
)

optimization_results.show(truncate=False

+--------------------------------+------------------+-------------+
|configuration                   |duree_secondes    |nb_partitions|
+--------------------------------+------------------+-------------+
|PostgreSQL sans cache           |0.6634457111358643|1            |
|Cache matérialisé               |0.4476473331451416|1            |
|Repartition(8, category) + cache|0.3778097629547119|8            |
|Écriture coalesce(1)            |0.5339391231536865|1            |
|Écriture repartition(1)         |0.2076869010925293|1            |
+--------------------------------+------------------+-------------+



### Tableau des mesures d'optimisation

| Configuration | Durée | Partitions | Observation |
|---|---:|---:|---|
| PostgreSQL sans cache | 0,663 s | 1 | Les traitements relisent les données depuis PostgreSQL |
| Cache matérialisé | 0,448 s | 1 | Les données sont réutilisées depuis le cache Spark |
| `repartition(8, "category")` + cache | 0,378 s | 8 | Un shuffle initial répartit les données, puis le cache accélère les réutilisations |
| Écriture avec `coalesce(1)` | 0,534 s | 1 | Réduction vers une partition sans redistribution complète |
| Écriture avec `repartition(1)` | 0,208 s | 1 | Redistribution complète vers une partition |

### Interprétation des mesures

La lecture PostgreSQL sans cache est la configuration la plus lente du workload, avec 0,663 seconde. Le DataFrame JDBC ne possède qu'une partition, ce qui limite également le parallélisme de la lecture.

Après matérialisation du cache, la durée passe à 0,448 seconde. Spark peut réutiliser les données stockées au lieu de relire PostgreSQL pour chaque traitement.

La configuration `repartition(8, "category")` avec cache obtient la meilleure durée pour le workload, avec 0,378 seconde. Les données sont distribuées sur huit partitions, ce qui permet davantage de parallélisme.

Dans cette exécution, `repartition(1)` est plus rapide que `coalesce(1)`. Ce résultat ne signifie pas que `repartition()` est toujours moins coûteux. Le résultat quotidien ne contient qu'environ 90 lignes : le coût du shuffle est donc très faible et les durées sont fortement influencées par l'échauffement de Spark, le cache du système et l'ordre des tests. Sur un grand DataFrame, `coalesce(1)` évite généralement davantage de transferts.

### Q3.4 — Différence entre `repartition` et `coalesce`

`repartition()` peut augmenter ou diminuer le nombre de partitions. Il provoque un shuffle complet afin de redistribuer et de rééquilibrer les données. Il est utile pour augmenter le parallélisme, corriger des partitions déséquilibrées ou préparer certaines jointures et agrégations.

`coalesce()` sert principalement à réduire le nombre de partitions. Il évite généralement une redistribution complète, ce qui le rend souvent moins coûteux. Cependant, les partitions obtenues peuvent être déséquilibrées.

Dans cette mesure, `repartition(1)` est plus rapide, mais le résultat écrit ne contient qu'environ 90 lignes. Cette mesure ponctuelle ne remet donc pas en cause le coût généralement supérieur du shuffle produit par `repartition()`.

### Q3.5 — Utilité et limites du cache

Le cache est utile lorsqu'un même DataFrame coûteux à charger ou à calculer est réutilisé plusieurs fois. Après sa matérialisation par une action, Spark peut lire les données depuis la mémoire ou le disque des executors plutôt que depuis PostgreSQL.

Dans notre mesure, le cache réduit la durée de 0,663 à 0,448 seconde. Le cache combiné à huit partitions réduit encore la durée à 0,378 seconde.

Le cache devient contre-productif lorsqu'un DataFrame n'est utilisé qu'une seule fois, lorsqu'il occupe trop de mémoire ou lorsque son coût de matérialisation dépasse les gains obtenus ensuite.

L'onglet Storage de Spark UI présente les RDD et DataFrames persistés, leur niveau de stockage, le nombre de partitions mises en cache et la quantité de mémoire ou de disque utilisée.

La méthode `unpersist()` libère les données mises en cache lorsqu'elles ne sont plus nécessaires.

In [85]:
hist.unpersist(blocking=True)
hist_cached.unpersist(blocking=True)
hist_repartitioned.unpersist(blocking=True)
spark.catalog.clearCache()

print("Caches Spark libérés")

Caches Spark libérés


## Réponses aux questions
- **Q0.1** : ...
- **Q0.2** : ...
- **Q3.1** : ...
- **Q3.2** : ...
- **Q3.3** : ...
- **Q3.4** : ...
- **Q3.5** : ...